# Indic quantization pilot on Kaggle

Measures per-language quantization damage (KL divergence, MILU accuracy, translation chrF++) for Gemma 3 4B in llama.cpp, with English and Indic importance-matrix calibration. Repository: https://github.com/ashutosh-rath02/indic-quant

**Notebook settings before running:** Accelerator *GPU T4 x2* or *GPU P100* (pick one and keep it for every session), Internet *on*, Persistence *none* is fine. Add a secret named `HF_TOKEN` under Add-ons > Secrets with a token whose account has accepted the terms of `google/gemma-3-4b-it`, `openlanguagedata/flores_plus` and `ai4bharat/MILU`.

For an unattended run use **Save Version > Save & Run All (Commit)**. Only `/kaggle/working` is kept when the run ends.

## Session 1 and session 2

The full pilot needs about 2 sessions of under 12 hours. Every step is resumable.

* **Session 1**: builds llama.cpp, downloads and converts the model, prepares data, builds the three imatrices, then streams variants (quantize, KLD, MILU, MT, delete) until the 10.5 hour budget is reached. It stops before a variant that would not finish and writes everything small to `/kaggle/working/iqp`.
* **Session 2**: open this notebook again (a new version), click **Add Input > Your Work > Notebook Output Files**, and attach the output of the session 1 version. It appears under `/kaggle/input/...`; the restore step finds it automatically. Session 2 copies the prepared data, imatrices, cached llama.cpp binaries and finished results, rebuilds only the F16 GGUF and base logits (about 30 to 45 minutes), and continues with the remaining variants. If a third session is needed, attach the session 2 output as well (or only it; the output always contains everything from earlier sessions).

Pick the same accelerator type in every session: base logits are recomputed per session, and mixing GPU types adds a small noise floor to the near-lossless variants.

## Configuration

In [ ]:
import os, time
os.environ.setdefault("IQP_SESSION_START", str(int(time.time())))  # time budget counts from here

REPO_URL = "https://github.com/ashutosh-rath02/indic-quant"
BRANCH = "main"
CONFIG = "configs/kaggle.yaml"          # or "configs/kaggle-qwen3-4b.yaml"
MAX_HOURS = 10.5                        # budget for this session; Kaggle stops at 12 h
RUN_TESTS = False                       # unit tests, about 2 minutes on 4 vCPUs

os.environ["IQP_WORK_DIR"] = "/tmp/iqp-work"            # scratch, large files
os.environ["IQP_OUTPUT_ROOT"] = "/kaggle/working/iqp"   # persisted
os.environ["IQP_RESTORE_FROM"] = "/kaggle/input"        # previous session outputs, if attached
os.environ["IQP_CUDA_ARCHS"] = "60;75"                  # P100 and T4
REPO = "/kaggle/working/indic-quant"

## Environment checks

In [ ]:
!nvidia-smi
!df -h /kaggle/working /tmp
!nproc && free -g
!(command -v nvcc && nvcc --version | tail -n 2) || (ls /usr/local/cuda/bin/nvcc && /usr/local/cuda/bin/nvcc --version | tail -n 2) || echo "ERROR: nvcc not found; choose a GPU accelerator"

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
assert os.environ["HF_TOKEN"].startswith("hf_"), "HF_TOKEN secret is missing or malformed"

## Code

In [ ]:
import subprocess

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

if os.path.isdir(REPO + "/.git"):
    sh(f"git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q --ff-only")
else:
    sh(f"git clone -q --branch {BRANCH} {REPO_URL} {REPO}")
sh(f"git -C {REPO} log -1 --oneline")
sh(f"pip install -q -r {REPO}/requirements.txt")
os.chdir(REPO)

In [ ]:
if RUN_TESTS:
    sh("python -m pytest -q")

In [ ]:
def iqp(*args):
    cmd = ["python", "-m", "iqp", *args, "--config", CONFIG]
    print("$", " ".join(cmd), flush=True)
    r = subprocess.run(cmd)
    if r.returncode != 0:
        raise RuntimeError(f"step failed: {' '.join(args)}; see {os.environ['IQP_OUTPUT_ROOT']}/logs")

iqp("plan")

## Restore the previous session (no-op in session 1)

In [ ]:
iqp("restore")
iqp("status")

## Build, download, convert, check data access

In [ ]:
iqp("setup")          # CUDA build, 15 to 30 min; skipped when restored from a cached build
iqp("download")
iqp("convert")        # F16 GGUF; HF weights are deleted afterwards
iqp("validate_data")  # fails here if a gated dataset is not accessible

## Data and imatrices (reused from session 1 when restored)

In [ ]:
iqp("prepare_data")
iqp("build_imatrix")

## Stream variants

Quantize, KLD on every language, MILU and MT for the selected variants, then delete the GGUF. Stops cleanly before a variant that would pass the budget.

In [ ]:
iqp("stream", "--max-hours", str(MAX_HOURS))

## Analysis and output

In [ ]:
for step in ("aggregate", "plot", "report", "bundle", "status"):
    iqp(step)
import json
st = json.load(open(os.environ["IQP_OUTPUT_ROOT"] + "/results/stream_status.json"))
print("complete:", st["complete"], "| remaining:", len(st.get("remaining", [])), st.get("remaining", []))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(os.environ["IQP_OUTPUT_ROOT"] + "/results/REPORT.md").read()))
!du -sh /kaggle/working/iqp /kaggle/working/iqp/*

## What is saved

`/kaggle/working/iqp` holds `IQP_OUTPUT.json`, `state/` (prepared data, imatrices, KLD plan, size records, timings, llama.cpp binaries), `results/` (raw JSON, CSVs, plots, REPORT.md) and `logs/`, plus `iqp-output.tar.gz` with the same content in one file. The repository clone under `/kaggle/working/indic-quant` is small and is saved too. Scratch under `/tmp/iqp-work` (GGUF files, base logits) is discarded.